# Self-Improvement & MLflow, Part 1

The code from [the article](https://opendatascience.com/self-improvement-vs-self-evolution-mlflow-part-1/), in article order. Each code cell is also saved as a file in this folder.

The snippets show the MLflow calls, not a complete agent: names such as `agent` or `run_agent`, and the helpers marked *our helper*, are yours to supply. Each cell says which names it expects. A snippet that continues an earlier one doesn't repeat its imports.

## What an agent can change


```text
prompt  ────▶  memory  ────▶  skill / code  ────▶  weights  ────▶  architecture
easiest to audit and roll back  ────────────────▶  hardest to audit and roll back
```

## Trace the agent

File: `support_agent.py`

Not defined here: `agent`, `policies` and `route_topic` are your own agent's code.

In [ ]:
import mlflow
from mlflow.entities import SpanType

mlflow.set_experiment("support-agent")
mlflow.openai.autolog()          # or langchain, anthropic, pydantic_ai, ...


@mlflow.trace(span_type=SpanType.TOOL)
def lookup_refund_policy(plan: str) -> str:
    return policies[plan]            # a child span, typed so judges can find it


@mlflow.trace(span_type=SpanType.AGENT)
def run_agent(question: str) -> str:
    mlflow.update_current_trace(tags={"topic": route_topic(question)})
    return agent.invoke(question)


## Log human feedback and an expectation

File: `log_feedback.py`

In [ ]:
from mlflow.entities import AssessmentSource, AssessmentSourceType

mlflow.log_feedback(
    trace_id="tr-7f3a9c",
    name="quality",
    value=False,
    rationale="Quoted last year's refund policy; the window changed in July.",
    source=AssessmentSource(
        source_type=AssessmentSourceType.HUMAN, source_id="maria"
    ),
)
mlflow.log_expectation(
    trace_id="tr-7f3a9c",
    name="expected_response",
    value="Refunds are available within 14 days of purchase.",
)


## Turn the feedback into scorers

File: `scorers.py`

In [ ]:
from mlflow.genai.scorers import Guidelines, scorer


@scorer
def cites_current_policy(outputs) -> bool:
    return "14 days" in str(outputs) and "30 days" not in str(outputs)


leads_with_answer = Guidelines(
    name="leads_with_answer",
    guidelines="The response must state the refund window in the first sentence. "
               "PASS if it does; FAIL if it opens with caveats.",
)


## Build a regression dataset from failing traces

File: `regression_dataset.py`

Not defined here: `experiment_id` is your experiment's id.

In [ ]:
from mlflow.genai.datasets import create_dataset

dataset = create_dataset(
    name="billing_regressions", experiment_id=[experiment_id]
)

failing = mlflow.search_traces(
    filter_string="feedback.quality = 'Fail' "
                  "AND expectation.expected_response IS NOT NULL",
    return_type="list",
)
dataset = dataset.merge_records(failing)


## Write a regression test

File: `test_refund_policy.py`

Not defined here: `leads_with_answer` comes from `scorers.py`.

In [ ]:
import mlflow
from mlflow.genai.scorers import RegexMatch

from support_agent import run_agent     # the production, instrumented entry point


@mlflow.test
def test_should_state_current_refund_policy():
    result = mlflow.genai.evaluate(
        predict_fn=run_agent,
        data=[{"inputs": {"question": "How long can I wait to request a refund?"}}],
        scorers=[
            RegexMatch(name="mentions_window", pattern=r"14 days"),
            leads_with_answer,
        ],
    )
    assert result.passed, result.reason
